# 1. dim_category population logic

In [0]:
from pyspark.sql import functions as F

category_source = (spark.table("azuresalesdatabricks.silver.products")
    .filter("is_current = true")
    .select("category", "subcategory")
    .distinct()
    .withColumn("category_id",
        F.concat(F.lit("CAT"), F.sha2(F.concat_ws("||", "category", "subcategory"), 256)))
    .withColumnRenamed("category", "category_name")
    .withColumnRenamed("subcategory", "subcategory_name")
)

category_source.createOrReplaceTempView("dim_category_batch")

spark.sql("""
    MERGE INTO azuresalesdatabricks.gold.dim_category AS target
    USING dim_category_batch AS source
    ON target.category_name = source.category_name
       AND target.subcategory_name = source.subcategory_name
    WHEN NOT MATCHED THEN INSERT *
""")

# 2. dim_customer population logic

## 2.1. Create the table with surrogate key

In [0]:
from pyspark.sql import functions as F

def merge_dim_customer(batch_df, batch_id):
    # Deterministic key: same customer_id + effective_start_date
    # ALWAYS produces the same hash, on any run, forever —
    # unlike monotonically_increasing_id(), which resets its
    # numbering scheme every time the job runs.
    with_key = batch_df.withColumn(
        "customer_key",
        F.concat(F.lit("CUSTK"),
                  F.sha2(F.concat_ws("||", "customer_id", "effective_start_date"), 256))
    )

    with_key.createOrReplaceTempView("dim_customer_batch")

    # Only INSERT — never UPDATE. Each (customer_id, effective_start_date)
    # combination is a specific, immutable historical version; once it
    # exists in gold, it never needs to change, only new versions get added.
    batch_df.sparkSession.sql("""
        MERGE INTO azuresalesdatabricks.gold.dim_customer AS target
        USING dim_customer_batch AS source
        ON target.customer_id = source.customer_id
           AND target.effective_start_date = source.effective_start_date
        WHEN NOT MATCHED THEN INSERT *
    """)

silver_customers_stream = spark.readStream.table("azuresalesdatabricks.silver.customers")

query = (silver_customers_stream.writeStream
    .foreachBatch(merge_dim_customer)
    .option("checkpointLocation", "abfss://gold@azuresales.dfs.core.windows.net/_checkpoints/dim_customer/")
    .trigger(availableNow=True)
    .start())
query.awaitTermination()

# 3. dim_product population logic

In [0]:
from pyspark.sql import functions as F

# ============================================================
# dim_product: pass-through from silver.products (ALL rows,
# not just is_current — same reasoning as dim_customer: facts
# need to range-join against historical product versions too),
# plus a stable hash-based surrogate key, plus a lookup into
# dim_category to resolve category_id.
# ============================================================
def merge_dim_product(batch_df, batch_id):

    # --------------------------------------------------------
    # Look up category_id from dim_category by matching on the
    # actual category/subcategory TEXT — dim_category doesn't
    # know about products, so this is the only way to connect
    # the two: match on the human-readable values they share.
    # --------------------------------------------------------
    category_lookup = spark.table("azuresalesdatabricks.gold.dim_category")

    with_category = (batch_df
        .join(category_lookup,
              (batch_df.category == category_lookup.category_name) &
              (batch_df.subcategory == category_lookup.subcategory_name),
              "left")
        .select(batch_df["*"], category_lookup["category_id"])
    )

    # --------------------------------------------------------
    # Stable, deterministic surrogate key — same pattern as
    # dim_customer: hash of (business key + version start date),
    # so re-running this never reassigns a different key to the
    # same product version.
    # --------------------------------------------------------
    with_key = with_category.withColumn(
        "product_key",
        F.concat(F.lit("PRODK"),
                 F.sha2(F.concat_ws("||", "product_id", "effective_start_date"), 256))
    )

    final = with_key.select(
        "product_key", "product_id", "product_name", "category_id",
        "brand", "unit_cost", "unit_price",
        "effective_start_date", "effective_end_date", "is_current"
    )

    final.createOrReplaceTempView("dim_product_batch")

    # Insert-only MERGE — each (product_id, effective_start_date)
    # version is immutable once it exists; only genuinely new
    # versions ever need to be added.
    batch_df.sparkSession.sql("""
        MERGE INTO azuresalesdatabricks.gold.dim_product AS target
        USING dim_product_batch AS source
        ON target.product_id = source.product_id
           AND target.effective_start_date = source.effective_start_date
        WHEN NOT MATCHED THEN INSERT *
    """)


silver_products_stream = spark.readStream.table("azuresalesdatabricks.silver.products")

query = (silver_products_stream.writeStream
    .foreachBatch(merge_dim_product)
    .option("checkpointLocation", "abfss://gold@azuresales.dfs.core.windows.net/_checkpoints/dim_product/")
    .trigger(availableNow=True)
    .start())
query.awaitTermination()

# 4. dim_store population logic

In [0]:
from pyspark.sql import functions as F

def merge_dim_store(batch_df, batch_id):
    with_key = batch_df.withColumn(
        "store_key",
        F.concat(F.lit("STOREK"),
                 F.sha2(F.concat_ws("||", "store_id", "effective_start_date"), 256))
    )

    final = with_key.select(
        "store_key", "store_id", "store_name", "region", "city", "state", "store_type",
        "effective_start_date", "effective_end_date", "is_current"
    )

    final.createOrReplaceTempView("dim_store_batch")

    batch_df.sparkSession.sql("""
        MERGE INTO azuresalesdatabricks.gold.dim_store AS target
        USING dim_store_batch AS source
        ON target.store_id = source.store_id
           AND target.effective_start_date = source.effective_start_date
        WHEN NOT MATCHED THEN INSERT *
    """)


silver_stores_stream = spark.readStream.table("azuresalesdatabricks.silver.stores")

query = (silver_stores_stream.writeStream
    .foreachBatch(merge_dim_store)
    .option("checkpointLocation", "abfss://gold@azuresales.dfs.core.windows.net/_checkpoints/dim_store/")
    .trigger(availableNow=True)
    .start())
query.awaitTermination()

# 5. dim_sales_rep population logic

In [0]:
from pyspark.sql import functions as F

def merge_dim_sales_rep(batch_df, batch_id):

    # --------------------------------------------------------
    # Look up store_key from dim_store by matching on store_id —
    # unlike the category lookup (which matched on text names),
    # here we can match on the actual business key, store_id,
    # since both silver.sales_reps and dim_store share it directly.
    # --------------------------------------------------------
    store_lookup = (spark.table("azuresalesdatabricks.gold.dim_store")
        .filter("is_current = true")   # a rep should link to the store's CURRENT version
        .select("store_id", "store_key")
    )

    with_store = (batch_df
        .join(store_lookup, on="store_id", how="left")
    )

    # Same stable hash-key pattern as every other dimension
    with_key = with_store.withColumn(
        "rep_key",
        F.concat(F.lit("REPK"),
                 F.sha2(F.concat_ws("||", "rep_id", "effective_start_date"), 256))
    )

    final = with_key.select(
        "rep_key", "rep_id", "rep_name", "email", "store_key", "region",
        "effective_start_date", "effective_end_date", "is_current"
    )

    final.createOrReplaceTempView("dim_sales_rep_batch")

    batch_df.sparkSession.sql("""
        MERGE INTO azuresalesdatabricks.gold.dim_sales_rep AS target
        USING dim_sales_rep_batch AS source
        ON target.rep_id = source.rep_id
           AND target.effective_start_date = source.effective_start_date
        WHEN NOT MATCHED THEN INSERT *
    """)


silver_reps_stream = spark.readStream.table("azuresalesdatabricks.silver.sales_reps")

query = (silver_reps_stream.writeStream
    .foreachBatch(merge_dim_sales_rep)
    .option("checkpointLocation", "abfss://gold@azuresales.dfs.core.windows.net/_checkpoints/dim_sales_rep/")
    .trigger(availableNow=True)
    .start())
query.awaitTermination()

# 6. fact_sales population logic

In [0]:
from pyspark.sql import functions as F

def merge_fact_sales(batch_df, batch_id):

    # --------------------------------------------------------
    # Step 1: order_items alone doesn't know the customer, store,
    # rep, or date — that's all on the order "header." Join them
    # together first, so each line item inherits its order's context.
    # --------------------------------------------------------
    orders = spark.table("azuresalesdatabricks.silver.sales_orders")

    items_with_order = (batch_df.alias("oi")
        .join(orders.alias("o"), "order_id", "inner")
        .select(
            F.col("oi.order_item_id"), 
            F.col("oi.order_id"),
            F.col("oi.product_id"), 
            F.col("oi.quantity"),
            F.col("oi.unit_price"), 
            F.col("oi.discount_pct"), 
            F.col("oi.line_total"),
            F.col("o.order_date"), 
            F.col("o.customer_id"),
            F.col("o.store_id"), 
            F.col("o.rep_id"), 
            F.col("o.order_status")
        )
    )

    # --------------------------------------------------------
    # Step 2: resolve customer_key via a RANGE join, not equality.
    # We need whichever version of the customer was TRUE on the
    # order's date — not whichever version is current today.
    # --------------------------------------------------------
    dim_customer = spark.table("azuresalesdatabricks.gold.dim_customer")
    with_customer = (items_with_order.alias("f")
        .join(dim_customer.alias("c"),
              (F.col("f.customer_id") == F.col("c.customer_id")) &
              (F.col("f.order_date") >= F.col("c.effective_start_date")) &
              ((F.col("f.order_date") < F.col("c.effective_end_date")) |
               F.col("c.effective_end_date").isNull()),
              "left")
        .select(F.col("f.*"), F.col("c.customer_key"))
    )

    # --------------------------------------------------------
    # Step 3: same range-join logic for product_key
    # --------------------------------------------------------
    dim_product = spark.table("azuresalesdatabricks.gold.dim_product")
    with_product = (with_customer.alias("f")
        .join(dim_product.alias("p"),
              (F.col("f.product_id") == F.col("p.product_id")) &
              (F.col("f.order_date") >= F.col("p.effective_start_date")) &
              ((F.col("f.order_date") < F.col("p.effective_end_date")) |
               F.col("p.effective_end_date").isNull()),
              "left")
        .select(F.col("f.*"), F.col("p.product_key"))
    )

    # --------------------------------------------------------
    # Step 4: store and rep rarely change, so — same simplification
    # as dim_sales_rep's own store lookup — just join to each one's
    # CURRENT version rather than a full range join.
    # --------------------------------------------------------
    dim_store = spark.table("azuresalesdatabricks.gold.dim_store").filter("is_current = true").select("store_id", "store_key")
    dim_rep   = spark.table("azuresalesdatabricks.gold.dim_sales_rep").filter("is_current = true").select("rep_id", "rep_key")

    with_store = with_product.join(dim_store, on="store_id", how="left")
    with_rep   = with_store.join(dim_rep, on="rep_id", how="left")

    # --------------------------------------------------------
    # Step 5: date_key is a simple, direct format-match — no
    # range join needed, dim_date has exactly one row per day.
    # --------------------------------------------------------
    final = (with_rep
        .withColumn("date_key", F.date_format("order_date", "yyyyMMdd"))
        .select("order_item_id", 
                "order_id", 
                "customer_key", 
                "product_key",
                "store_key", 
                "rep_key", 
                "date_key", 
                "quantity", 
                "unit_price",
                "discount_pct", 
                "line_total", 
                "order_status")
    )

    final.createOrReplaceTempView("fact_sales_batch")

    # --------------------------------------------------------
    # Step 6: insert-only MERGE. Facts are immutable once recorded —
    # a sale doesn't get "updated" the way a dimension does.
    # --------------------------------------------------------
    batch_df.sparkSession.sql("""
        MERGE INTO azuresalesdatabricks.gold.fact_sales AS target
        USING fact_sales_batch AS source
        ON target.order_item_id = source.order_item_id
        WHEN NOT MATCHED THEN INSERT *
    """)


silver_items_stream = spark.readStream.table("azuresalesdatabricks.silver.order_items")

query = (silver_items_stream.writeStream
    .foreachBatch(merge_fact_sales)
    .option("checkpointLocation", "abfss://gold@azuresales.dfs.core.windows.net/_checkpoints/fact_sales/")
    .trigger(availableNow=True)
    .start())
query.awaitTermination()

# 7. fact_returns population logic

In [0]:
from pyspark.sql import functions as F

def merge_fact_returns(batch_df, batch_id):

    # --------------------------------------------------------
    # date_key: same direct-format trick as fact_sales — no join
    # needed, since dim_date's key is derivable straight from the
    # raw date using the identical formatting rule.
    # --------------------------------------------------------
    final = (batch_df
        .withColumn("date_key", F.date_format("return_date", "yyyyMMdd"))
        .select("return_id", "order_item_id", "date_key", "return_reason", "refund_amount")
    )

    final.createOrReplaceTempView("fact_returns_batch")

    # Insert-only MERGE — same idempotent pattern as fact_sales.
    # A recorded return doesn't get "updated," only new ones added.
    batch_df.sparkSession.sql("""
        MERGE INTO azuresalesdatabricks.gold.fact_returns AS target
        USING fact_returns_batch AS source
        ON target.return_id = source.return_id
        WHEN NOT MATCHED THEN INSERT *
    """)


silver_returns_stream = spark.readStream.table("azuresalesdatabricks.silver.returns")

query = (silver_returns_stream.writeStream
    .foreachBatch(merge_fact_returns)
    .option("checkpointLocation", "abfss://gold@azuresales.dfs.core.windows.net/_checkpoints/fact_returns/")
    .trigger(availableNow=True)
    .start())
query.awaitTermination()